# A model that stays fitted, and stays the same object

This notebook fits a small model (linear regression via gradient descent, 8 million
synthetic samples, 140 epochs -- a real, CPU-bound loop, not a placeholder) once, then reuses
it across several independent cell executions.

Two things are true here that would not be true if every cell relaunched a fresh process:

1. **It's the literal same object.** The training data's memory address is identical in
   every cell below, with zero copying. A value nudged in one cell is visible, unmodified by
   anything else, when read back from a completely separate, independently compiled cell
   later on.
2. **Fitting it happens exactly once**, however long that takes. Here it's about a second and
   a half -- on a real model with a real dataset it could be minutes.

gocell compiles each cell as a Go plugin loaded into the *same* long-lived kernel process, so
both of these fall out of the architecture for free. For comparison,
[gonb](https://github.com/janpfeifer/gonb) does not keep a single persistent process running
across cells -- by its own documentation, it recompiles and reruns an accumulated program for
every cell. Whether it manages to skip redoing expensive prior work automatically isn't
something this notebook claims to know either way; what *is* certain is point 1 above --
mutating a live object in place and reading that exact mutation back from a later,
independent cell -- which isn't achievable at all without a persistent process, regardless of
how cleverly re-execution is cached.

In [ ]:
import (
	"math"
	"time"
)

start := time.Now()

n := 8_000_000
xs := make([]float64, n)
ys := make([]float64, n)
for i := 0; i < n; i++ {
	x := float64(i) / float64(n)
	xs[i] = x
	ys[i] = 3.1*x + 0.7 + 0.01*math.Sin(x*50)
}

w, b := 0.0, 0.0
lr := 0.5
epochs := 140
for epoch := 0; epoch < epochs; epoch++ {
	var gw, gb float64
	for i := 0; i < n; i++ {
		pred := w*xs[i] + b
		errv := pred - ys[i]
		gw += errv * xs[i]
		gb += errv
	}
	w -= lr * gw / float64(n)
	b -= lr * gb / float64(n)
}

fmt.Printf("Model fit in %s -> w=%.4f b=%.4f (target w=3.1 b=0.7)\n", time.Since(start), w, b)

## Reuse it whenever you like -- no refit

This is a fresh, independently compiled cell. It only reads `w` and `b` back from the shared
state -- no setup cell to re-run first.

In [ ]:
predictStart := time.Now()
prediction := w*0.42 + b
fmt.Printf("predict(0.42) = %.4f in %s\n", prediction, time.Since(predictStart))

## Same object, no reload

Run this cell whenever you like -- the address of the training data matches exactly, every time.

In [ ]:
fmt.Printf("xs address: %p\n", xs)

In [ ]:
fmt.Printf("still the same xs: %p\n", xs)

## Mutating state in place

Nudge the fitted weight by hand -- a plain in-place mutation, nothing gocell-specific about
this line by itself.

In [ ]:
w = w * 1.1
fmt.Printf("nudged w to %.4f\n", w)

Now, in a brand new cell -- a fresh compile, a fresh plugin, no reference back to the
previous one in this notebook's text -- read it back:

In [ ]:
// A bare expression as the last line of a cell is auto-displayed as the cell's result,
// the equivalent of Jupyter's Out[n] -- no fmt.Println needed.
w